In [ ]:
# Requirements
%pip install -r requirements.txt -q

# Título

Apresentação projeto

In [ ]:
# Imports
import mne
import numpy as np
import pandas as pd
from pathlib import Path
import sys

# Add the Inner Speech Dataset repository to the system path
sys.path.append(str(Path.cwd() / "Python_Processing"))

# Import custom functions from the Inner Speech Dataset repository
from lib.data_extractions import (
    extract_data_from_subject,
    load_events
)
from lib.data_processing import (
    select_time_window,
    transform_for_classificator,
)
from lib.download_helper import (
    _clone_inner_speech,
    get_derivatives,
)
from lib.utils import (
    sub_name, 
    unify_names,
)

In [ ]:
# Some basic definitions
# EEG sampling rate (256 samples per second)
fs = 256

# Select the time window of interest (in seconds)
# The cue appears at t=0, action period is between 1.5s and 3.5s
t_start = 1.5  # Start time after cue
t_end = 3.5  # End time after cue

# List with every subject (S1-S10)
N_S = list(range(1, 11))

# sub-09 ses-01 and sub-10 ses-03 are truncated in the OpenNeuro derivatives
EXCLUDE = {9, 10}
SUBJECTS = [i for i in N_S if i not in EXCLUDE]

data_dir = Path.cwd().parent
root_dir = data_dir / "ds003626"

## Download dos dados

Para fazer download dos dados, descomente o código abaixo. Os dados utilizados estão disponíveis [aqui](https://openneuro.org/datasets/ds003626).

In [ ]:
# Uncomment to download processed data
# for i in N_S:
#     get_derivatives(
#         dataset_path=data_dir,
#         subjects=f"{i:02d}",
#         sessions=None,
#         file_types=["eeg","events"],
#         verbose=True,
#         progress=True,
#     )

## Carregamento dos dados

Os dados de cada participante estão separados em dois arrays:
- **X**: Sinais EEG [trials × channels × time points]
- **Y**: Rótulos especificando o evento [trials × 4]:
  - Coluna 0: Timestamp
  - COluna 1: Classe (0=up, 1=down, 2=right, 3=left)
  - Coluna 2: Condição (0=pronounced, 1=inner speech, 2=visualized)
  - Coluna 3: Sessão

In [ ]:
# Loads all data from a single subject
def load_subject(root_dir: Path, n_s: int, datatype="eeg"):
    num_s = sub_name(n_s)
    x, y = [], []

    for n_b in (1, 2, 3):
        file_name = f"{root_dir}/derivatives/{num_s}/ses-0{n_b}/{num_s}_ses-0{n_b}_{datatype}-epo.fif"

        try:
            ep = mne.read_epochs(file_name, verbose="WARNING")
            data = ep.get_data()
        except Exception as e:
            print(f"Skipping {num_s} ses-0{n_b}: {type(e).__name__}")
            continue

        ev = pd.DataFrame(load_events(root_dir, n_s, n_b),
                          columns=["timestamp", "direction", "condition", "session"])
        
        ev["subject"] = n_s

        x.append(data)
        y.append(ev)

        ch_names, times = ep.ch_names, ep.times
        
        print(f"S_{n_s} ses-0{n_b}: {len(data)} trials")

    X = np.concatenate(x)
    Y = pd.concat(y, ignore_index=True)

    return X, Y, ch_names, times

### Salvando dados em arrays numpy

Carrega os dados de cada participante e os salva em arquivos .npy

In [ ]:
# Loads each subject's data and saves it as np arrays
for s in SUBJECTS:
    X, Y, ch_names, times = load_subject(root_dir, s)
    print(f"S_{s}: X = {X.shape}, Y = {Y.shape}\n")
    np.save(f"data/S{s}_f", np.asarray(X))
    np.save(f"data/S{s}_y", np.asarray(Y))
    np.save(f"data/S{s}_channels", np.asarray(ch_names))
    np.save(f"data/S{s}_times", np.asarray(times))

Os arquivos salvos são utilizados para construir um único array numpy, separando as classes e trials em duas dimensões diferentes. Serão considerados os dados sob a condição 1 - Inner Speech para todas as classes disponíveis.

In [ ]:
# Saves the file in (classes, trials, channels, timepoints) format

# X_cls = EEG data separated by CLASSES
# subj = mapping of X_cls[i,j] to the subject it came from (subj[i,j]) 
# ses = session of X_cls[i,j] to the session it came from

CLASSES = {0: "up", 1: "down", 2: "right", 3: "left"}
COND = 1 # 0 = pronounced, 1 = inner, 2 = visualized
rng = np.random.default_rng(0)

class_arr = {c: [] for c in CLASSES}
class_sbj = {c: [] for c in CLASSES}
class_ses = {c: [] for c in CLASSES}

for s in SUBJECTS:
    X = np.load(f"data/S{s}_f.npy")
    Y = np.load(f"data/S{s}_y.npy")

    for c in CLASSES:
        idx = np.where((Y[:, 1] == c) & (Y[:, 2] == COND))[0]

        class_arr[c].append(np.asarray(X[idx], dtype=np.float32))
        class_sbj[c].append(np.full(len(idx), s))
        class_ses[c].append(Y[idx, 3])

class_arr = {c: np.concatenate(v) for c, v in class_arr.items()}
class_sbj = {c: np.concatenate(v) for c, v in class_sbj.items()}
class_ses = {c: np.concatenate(v) for c, v in class_ses.items()}

counts = {CLASSES[c]: len(v) for c, v in class_arr.items()}
print(f"Counts for each class: {counts}")
n = min(counts.values())

X_cls, subj, ses = [], [], []
for c in CLASSES:
    keep = np.sort(rng.choice(len(class_arr[c]), n, replace=False))
    X_cls.append(class_arr[c][keep])
    subj.append(class_sbj[c][keep])
    ses.append(class_ses[c][keep])

X_cls = np.stack(X_cls)
subj = np.stack(subj)
ses = np.stack(ses)

np.save("data/X_cls.npy", X_cls)
np.save("data/subj.npy", subj)
np.save("data/ses.npy", ses)

print(f"Saved in format {X_cls.shape}")

## Visualização dos Dados

In [ ]:
import matplotlib.pyplot as plt

CLASSES = {0: "up", 1: "down", 2: "right", 3: "left"}

times = np.load("data/S1_times.npy")
ch_names = np.load("data/S1_channels.npy")
X_cls = np.load("data/X_cls.npy", mmap_mode="r")

# x = np.asarray(X_cls[0, :50])
# print("std:", x.std(), "| max abs:", np.abs(x).max())

subj = np.load("data/subj.npy")

print(times[0], times[-1], len(times)) # check time range

def plot_channels(classes=(0, 1, 2, 3), subject=None, tmin=None, tmax=None, ncols=8, save_as=None):
    keep = np.ones(len(times), dtype=bool)

    if tmin is not None: keep &= times >= tmin
    if tmax is not None: keep &= times <= tmax
    
    t = times[keep]
    n_ch = X_cls.shape[2]
    nrows = int(np.ceil(n_ch / ncols))
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.0 * ncols, 2.0 * nrows), sharex=True, sharey=True)

    for c in classes:
        trials = X_cls[c] if subject is None else X_cls[c][subj[c] == subject]
        mean = trials[:, :, keep].mean(axis=0) * 1e6 # convert volts to microvolts
        
        for ch, ax in enumerate(axes.flat[:n_ch]):
            ax.plot(t, mean[ch], lw=0.3, label=CLASSES[c])

    for ch, ax in enumerate(axes.flat[:n_ch]):
        ax.set_title(str(ch_names[ch]), fontsize=7, pad=2)
        ax.tick_params(labelsize=6)
        ax.axvline(0, color="black", lw=0.4, alpha=0.4)
    
    for ax in axes.flat[n_ch:]:
        ax.axis("off")

    axes.flat[0].legend(fontsize=6, loc="upper right")
    who = "all subjects" if subject is None else f"subject {subject}"
    fig.legend(
        labels=[CLASSES[c] for c in classes],
        loc="upper center",
        bbox_to_anchor=(0.5, 1.0),
        ncol=len(classes),
        fontsize=10,
        frameon=False
    )
    fig.suptitle(f"Mean EEG per channel ({who}), µV vs. time (s)", y=1.002)
    fig.tight_layout()

    if save_as:
        fig.savefig(save_as, dpi=600, bbox_inches="tight")
    
    plt.show()

In [ ]:
for i in SUBJECTS:
    plot_channels(subject=i, save_as=f"figures/s{i}")

In [ ]:
# Debugging
X_cls = np.load("data/X_cls.npy")
subj = np.load("data/subj.npy")
ses = np.load("data/ses.npy")

print(X_cls.shape, subj.shape, ses.shape) # expected (4, 439, 128, 1153) (4, 439) (4, 439)
print(np.unique(subj[0], return_counts=True)) # expected 50 60 45 60 60 54 60 50 for subjects 1, 2, ..., 8
print(np.unique(ses, return_counts=True)) # sessions 1, 2, 3 only
print((subj == subj[0]).all()) # same subject order in every class